In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
pm_df = con.execute("SELECT * FROM opspatial WHERE total_reception_events >= 15 AND total_release_events >= 15 AND position != 'Goalkeeper'").df()
con.close()

In [ ]:
pm_df["avg_reception_simpnorm_y"] = np.where(
    pm_df["average_reception_y"] <= 40,
    pm_df["average_reception_y"],
    80 - pm_df["average_reception_y"],
)

pm_df["avg_release_simpnorm_y"] = np.where(
    pm_df["average_release_y"] <= 40,
    pm_df["average_release_y"],
    80 - pm_df["average_release_y"],
)

pm_df["avg_reception_simpnorm_y_abs"] = pm_df["avg_reception_simpnorm_y"].abs()
pm_df["avg_release_simpnorm_y_abs"] = pm_df["avg_release_simpnorm_y"].abs()

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

In [ ]:
pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

pm_k_range = range(2, 11)
pm_inertia = []
pm_silhouette_scores = []

for n_clusters in pm_k_range:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = model.fit_predict(pm_X_scaled)

    pm_inertia.append(model.inertia_)
    pm_silhouette_scores.append(
        silhouette_score(
            pm_X_scaled,
            labels,
            sample_size=min(5000, len(pm_X_scaled)),
            random_state=42,
        )
    )

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(pm_k_range), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(pm_k_range), pm_silhouette_scores, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)
cluster_data = pm_df.loc[pm_X.index].copy()
cluster_counts = []

for n_clusters in [3, 4, 5, 6]:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_data[f"cluster_k{n_clusters}"] = model.fit_predict(pm_X_scaled)
    pm_df[f"cluster_k{n_clusters}"] = cluster_data[f"cluster_k{n_clusters}"]

    counts = (
        cluster_data.groupby(f"cluster_k{n_clusters}")
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={f"cluster_k{n_clusters}": "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    cluster_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", n_clusters)
    cluster_col = f"cluster_k{n_clusters}"

    for cluster in range(n_clusters):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color_map(cluster), label=f"Cluster {cluster}")
        for cluster in range(n_clusters)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial clusters (k={n_clusters})")
    plt.tight_layout()
    plt.show()

cluster_count_df = pd.concat(cluster_counts, ignore_index=True)
display(cluster_count_df)

In [ ]:
pm_X = pm_df[feature_cols].dropna().copy()

pm_k_range = range(2, 11)
pm_inertia = []
pm_silhouette_scores = []

for n_clusters in pm_k_range:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = model.fit_predict(pm_X)

    pm_inertia.append(model.inertia_)
    pm_silhouette_scores.append(
        silhouette_score(
            pm_X,
            labels,
            sample_size=min(5000, len(pm_X)),
            random_state=42,
        )
    )

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(pm_k_range), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(pm_k_range), pm_silhouette_scores, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
pm_X = pm_df[feature_cols].dropna().copy()
cluster_data = pm_df.loc[pm_X.index].copy()
cluster_counts = []

for n_clusters in [3, 4, 5, 6, 7, 8]:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_data[f"cluster_k{n_clusters}"] = model.fit_predict(pm_X)
    pm_df[f"cluster_k{n_clusters}"] = cluster_data[f"cluster_k{n_clusters}"]

    counts = (
        cluster_data.groupby(f"cluster_k{n_clusters}")
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={f"cluster_k{n_clusters}": "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    cluster_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", n_clusters)
    cluster_col = f"cluster_k{n_clusters}"

    for cluster in range(n_clusters):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color_map(cluster), label=f"Cluster {cluster}")
        for cluster in range(n_clusters)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial clusters (k={n_clusters})")
    plt.tight_layout()
    plt.show()

cluster_count_df = pd.concat(cluster_counts, ignore_index=True)
display(cluster_count_df)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage

pm_X = pm_df[feature_cols].dropna().copy()
cluster_data = pm_df.loc[pm_X.index].copy()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

# Ward linkage uses all rows; truncate only the display so the tree remains readable.
linkage_matrix = linkage(pm_X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=40,
    show_contracted=True,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title("Player-match hierarchy (Ward linkage)")
ax.set_xlabel("Player-match groups (truncated to the last 40 merges)")
ax.set_ylabel("Ward distance")
plt.tight_layout()
plt.show()

hierarchy_counts = []
for n_clusters in [3, 4, 5, 6, 7, 8, 9]:
    cluster_col = f"hierarchy_k{n_clusters}"
    cluster_data[cluster_col] = fcluster(
        linkage_matrix, t=n_clusters, criterion="maxclust"
    )
    pm_df.loc[cluster_data.index, cluster_col] = cluster_data[cluster_col]

    labels = sorted(cluster_data[cluster_col].unique())
    counts = (
        cluster_data.groupby(cluster_col)
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={cluster_col: "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    hierarchy_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", len(labels))

    for cluster_index, cluster in enumerate(labels):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster_index)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D(
            [0], [0], marker="o", color="none",
            markerfacecolor=color_map(cluster_index), label=f"Cluster {cluster}"
        )
        for cluster_index, cluster in enumerate(labels)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial hierarchy (k={n_clusters})")
    plt.tight_layout()
    plt.show()

display(pd.concat(hierarchy_counts, ignore_index=True))

In [ ]:
pm_X = pm_df[feature_cols].dropna().copy()
cluster_data = pm_df.loc[pm_X.index].copy()

# Ward linkage uses all rows; truncate only the display so the tree remains readable.
linkage_matrix = linkage(pm_X, method="ward")

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=40,
    show_contracted=True,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title("Player-match hierarchy (Ward linkage)")
ax.set_xlabel("Player-match groups (truncated to the last 40 merges)")
ax.set_ylabel("Ward distance")
plt.tight_layout()
plt.show()

hierarchy_counts = []
for n_clusters in [3, 4, 5, 6, 7, 8, 9]:
    cluster_col = f"hierarchy_k{n_clusters}"
    cluster_data[cluster_col] = fcluster(
        linkage_matrix, t=n_clusters, criterion="maxclust"
    )
    pm_df.loc[cluster_data.index, cluster_col] = cluster_data[cluster_col]

    labels = sorted(cluster_data[cluster_col].unique())
    counts = (
        cluster_data.groupby(cluster_col)
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={cluster_col: "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    hierarchy_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", len(labels))

    for cluster_index, cluster in enumerate(labels):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster_index)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D(
            [0], [0], marker="o", color="none",
            markerfacecolor=color_map(cluster_index), label=f"Cluster {cluster}"
        )
        for cluster_index, cluster in enumerate(labels)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial hierarchy (k={n_clusters})")
    plt.tight_layout()
    plt.show()

display(pd.concat(hierarchy_counts, ignore_index=True))

In [ ]:
position_data = pm_df.dropna(
    subset=[
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
        "position",
    ]
).copy()
positions = sorted(position_data["position"].unique())
position_colors = plt.get_cmap("tab20", len(positions))

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
fig, ax = pitch.draw(figsize=(12, 8))

for position_index, position in enumerate(positions):
    position_rows = position_data[position_data["position"] == position]
    color = position_colors(position_index)
    pitch.lines(
        position_rows["average_reception_x"],
        position_rows["average_reception_y"],
        position_rows["average_release_x"],
        position_rows["average_release_y"],
        color=color,
        alpha=0.25,
        lw=0.7,
        ax=ax,
    )
    pitch.scatter(
        position_rows["average_reception_x"],
        position_rows["average_reception_y"],
        color=color,position_data = pm_df.dropna(
    subset=[
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
        "position",
    ]
).copy()
positions = sorted(position_data["position"].unique())
position_colors = plt.get_cmap("tab20", len(positions))

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
fig, ax = pitch.draw(figsize=(12, 8))

for position_index, position in enumerate(positions):
    position_rows = position_data[position_data["position"] == position]
    color = position_colors(position_index)
    pitch.lines(
        position_rows["average_reception_x"],
        position_rows["average_reception_y"],
        position_rows["average_release_x"],
        position_rows["average_release_y"],
        color=color,
        alpha=0.25,
        lw=0.7,
        ax=ax,
    )
    pitch.scatter(
        position_rows["average_reception_x"],
        position_rows["average_reception_y"],
        color=color,
        marker="o",
        s=24,
        ax=ax,
    )
    pitch.scatter(
        position_rows["average_release_x"],
        position_rows["average_release_y"],
        color=color,
        marker="^",
        s=28,
        ax=ax,
    )

position_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=position_colors(index), label=position
    )
    for index, position in enumerate(positions)
]
position_handles.extend(
    [
        Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
        Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
    ]
)
ax.legend(handles=position_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Average reception-to-release locations by position")
plt.tight_layout()
plt.show()
        marker="o",
        s=24,
        ax=ax,
    )
    pitch.scatter(
        position_rows["average_release_x"],
        position_rows["average_release_y"],
        color=color,
        marker="^",
        s=28,
        ax=ax,
    )

position_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=position_colors(index), label=position
    )
    for index, position in enumerate(positions)
]
position_handles.extend(
    [
        Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
        Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
    ]
)
ax.legend(handles=position_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Average reception-to-release locations by position")
plt.tight_layout()
plt.show()

In [ ]:
position_data = pm_df.dropna(
    subset=[
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
        "position",
    ]
).copy()
positions = sorted(position_data["position"].unique())

ncols = 3
nrows = (len(positions) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
position_colors = plt.get_cmap("tab20", len(positions))

for index, position in enumerate(positions):
    ax = axes.flat[index]
    pitch.draw(ax=ax)

    rows = position_data[position_data["position"] == position]
    color = position_colors(index)

    pitch.lines(
        rows["average_reception_x"],
        rows["average_reception_y"],
        rows["average_release_x"],
        rows["average_release_y"],
        color=color,
        alpha=0.3,
        lw=0.8,
        ax=ax,
    )
    pitch.scatter(
        rows["average_reception_x"],
        rows["average_reception_y"],
        color=color,
        marker="o",
        s=24,
        ax=ax,
    )
    pitch.scatter(
        rows["average_release_x"],
        rows["average_release_y"],
        color=color,
        marker="^",
        s=28,
        ax=ax,
    )
    ax.set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
        Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
    ],
    loc="upper center",
    ncol=2,
)
fig.suptitle("Average reception-to-release locations by position", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()